# Step 1: Which data sources cover my anchor areas?

Runs on Kaggle or Colab (nothing touches your own computer). It answers, for each anchor area:
how many places and reviews exist in Google Local, how many businesses in Yelp, how complete OSM looks,
and shows sample review text so you can judge whether it describes the area.

Anchor coordinates below are approximate. Check them on a map and adjust.

In [ ]:
!pip -q install osmnx pandas requests

In [ ]:
import glob, gzip, json, os
import numpy as np
import pandas as pd
import requests

RADIUS_M = 800  # ~10 minute walk

# (lat, lon) approximate centers. Verify on a map and edit.
AREAS = {
    "Rice Village":    (29.7172, -95.4167),
    "Montrose":        (29.7430, -95.3900),
    "Memorial Park":   (29.7634, -95.4410),
    # Add target-city areas here to compare coverage, e.g.:
    # "Austin - South Congress": (30.2490, -97.7500),
}

STATE = "Texas"
# Paste the direct links from the UCSD Google Local dataset page (McAuley lab).
# Use the per-state metadata file and per-state review file for your state.
GOOGLE_META_URL = ""
GOOGLE_REVIEW_URL = ""

# Where the Yelp files live (Kaggle: attach a Yelp dataset; Colab: Drive path).
YELP_DIR = "/kaggle/input"

REVIEWS_PER_PLACE_CAP = 40
MAX_REVIEW_LINES = None  # set e.g. 5_000_000 for a quick partial run

In [ ]:
def dist_m(lat, lon, lat0, lon0):
    p = np.pi / 180
    a = (np.sin((lat - lat0) * p / 2) ** 2
         + np.cos(lat0 * p) * np.cos(lat * p) * np.sin((lon - lon0) * p / 2) ** 2)
    return 2 * 6371000 * np.arcsin(np.sqrt(a))


def nearest_area(lat, lon):
    for name, (a_lat, a_lon) in AREAS.items():
        if dist_m(lat, lon, a_lat, a_lon) <= RADIUS_M:
            return name
    return None


def stream_jsonl_gz(url, max_lines=None):
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        with gzip.open(r.raw, "rt", encoding="utf-8") as f:
            for i, line in enumerate(f):
                if max_lines and i >= max_lines:
                    break
                try:
                    yield json.loads(line)
                except json.JSONDecodeError:
                    continue

## Google Local: places near each anchor

In [ ]:
google_places = pd.DataFrame()
if GOOGLE_META_URL:
    rows = []
    for i, m in enumerate(stream_jsonl_gz(GOOGLE_META_URL)):
        if i == 0:
            print("Raw first metadata row (check field names):")
            print(json.dumps(m, indent=2)[:800])
        lat, lon = m.get("latitude"), m.get("longitude")
        if lat is None or lon is None:
            continue
        area = nearest_area(lat, lon)
        if area:
            rows.append({
                "area": area,
                "gmap_id": m.get("gmap_id"),
                "name": m.get("name"),
                "category": m.get("category"),
                "avg_rating": m.get("avg_rating"),
                "num_of_reviews": m.get("num_of_reviews"),
            })
    google_places = pd.DataFrame(rows)
    print(google_places.groupby("area").agg(
        places=("gmap_id", "count"),
        total_reviews=("num_of_reviews", "sum")))
else:
    print("Set GOOGLE_META_URL above to run this section.")

## Google Local: review text near each anchor

This streams the whole state review file (several GB), so it can take a while. Use `MAX_REVIEW_LINES` for a quick partial run.

In [ ]:
google_reviews = pd.DataFrame()
if GOOGLE_REVIEW_URL and len(google_places):
    id_to_area = dict(zip(google_places.gmap_id, google_places.area))
    per_place = {}
    rows = []
    for i, r in enumerate(stream_jsonl_gz(GOOGLE_REVIEW_URL, MAX_REVIEW_LINES)):
        if i == 0:
            print("Raw first review row (check field names):")
            print(json.dumps(r, indent=2)[:600])
        gid = r.get("gmap_id")
        if gid not in id_to_area:
            continue
        text = r.get("text") or ""
        if len(text) < 60 or per_place.get(gid, 0) >= REVIEWS_PER_PLACE_CAP:
            continue
        per_place[gid] = per_place.get(gid, 0) + 1
        rows.append({"area": id_to_area[gid], "gmap_id": gid,
                     "rating": r.get("rating"), "text": text})
        if i and i % 2_000_000 == 0:
            print(f"{i:,} lines scanned, {len(rows):,} reviews kept")
    google_reviews = pd.DataFrame(rows)
    print(google_reviews.groupby("area").size().rename("reviews_with_text (capped per place)"))
else:
    print("Needs GOOGLE_REVIEW_URL and the places table from the previous cell.")

In [ ]:
# Read these. Do they describe the AREA, or just the business?
if len(google_reviews):
    for area, g in google_reviews.groupby("area"):
        print(f"\n===== {area} =====")
        for t in g.sample(min(8, len(g)), random_state=1).text:
            print("-", t[:300].replace("\n", " "))

## Yelp: city coverage and businesses near each anchor

In [ ]:
yelp_counts = {}

# Metro boxes (west, south, east, north), approximate - adjust. Used to catch suburbs listed under other city names.
METRO_BOXES = {
    "Houston metro": (-95.95, 29.45, -94.95, 30.15),
    # "Austin metro": (-98.10, 30.00, -97.40, 30.65),
}

paths = glob.glob(os.path.join(YELP_DIR, "**", "*business.json"), recursive=True)
if paths:
    b = pd.read_json(paths[0], lines=True)
    print("Top cities in Yelp (raw city strings):")
    print(b.groupby(["city", "state"]).size().sort_values(ascending=False).head(25))

    # Coordinate-based metro coverage: does not depend on the city string
    for name, (w, s, e, n) in METRO_BOXES.items():
        inbox = b[(b.longitude >= w) & (b.longitude <= e) & (b.latitude >= s) & (b.latitude <= n)]
        print()
        print(f"{name}: {len(inbox):,} Yelp businesses inside the box")
        if len(inbox):
            print("City strings used inside the box:")1
            print(inbox.groupby(["city", "state"]).size().sort_values(ascending=False).head(15))

    # Neighborhood-level: businesses within RADIUS_M of each anchor (coordinates only)
    for area, (a_lat, a_lon) in AREAS.items():
        d = dist_m(b.latitude.values, b.longitude.values, a_lat, a_lon)
        yelp_counts[area] = int((d <= RADIUS_M).sum())
    print()
    print("Yelp businesses within radius:", yelp_counts)
else:
    print(f"No *business.json found under {YELP_DIR}. Attach the Yelp dataset or fix YELP_DIR.")

## OSM: how complete does the map look around each anchor?

In [ ]:
import osmnx as ox

osm_counts = {}
tags = {"amenity": ["restaurant", "cafe", "bar", "pub", "fast_food"]}
for area, (a_lat, a_lon) in AREAS.items():
    try:
        gdf = ox.features_from_point((a_lat, a_lon), tags, dist=RADIUS_M)
        osm_counts[area] = {
            "food_pois": len(gdf),
            "with_cuisine_tag": int(gdf["cuisine"].notna().sum()) if "cuisine" in gdf else 0,
            "with_brand_tag": int(gdf["brand"].notna().sum()) if "brand" in gdf else 0,
        }
    except Exception as e:
        osm_counts[area] = {"error": str(e)[:80]}
pd.DataFrame(osm_counts).T

## Summary: which source is worth building on?

In [ ]:
summary = pd.DataFrame(index=list(AREAS))
if len(google_places):
    summary["google_places"] = google_places.groupby("area").size()
    summary["google_total_reviews"] = google_places.groupby("area").num_of_reviews.sum()
if len(google_reviews):
    summary["google_reviews_with_text"] = google_reviews.groupby("area").size()
if yelp_counts:
    summary["yelp_businesses"] = pd.Series(yelp_counts)
if osm_counts:
    summary["osm_food_pois"] = pd.Series({k: v.get("food_pois") for k, v in osm_counts.items()})
summary